# A radial gradient echo, one spoke at a time

`sc.modules.RadialReadout` is **one spoke**: a prephaser, a readout gradient and an ADC, already
oriented. It knows its own trajectory — where `k = 0` falls among its samples, how far it reaches,
how far apart the samples are — and nothing at all about how many spokes there are or in what
order they should be played.

That division is the point of this notebook. There is deliberately **no `RadialGRE` class**: an
acquisition is the spoke plus an angle schedule, and a schedule is a list you write.

```text
RadialReadout   owns one spoke
the caller      owns how many, at which angles, in what order
```

**Output:** two `.seq` files — an equal-increment acquisition and a golden-angle one — from one
instance of the readout.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pypulseq as pp

import seqcraft as sc

opts = pp.Opts(
    max_grad=28, grad_unit='mT/m',
    max_slew=120, slew_unit='T/m/s',
    B0=3.0,
    rf_dead_time=100e-6,
    rf_ringdown_time=20e-6,
    adc_dead_time=10e-6,
)

FOV_MM, MATRIX, THICKNESS_MM = 260.0, 64, 5.0
DWELL_S, FLIP_DEG = 20e-6, 10.0
SPOKES, DUMMIES, TR_S = 64, 8, 10e-3

SEQ_DIR = Path('seq')
SEQ_DIR.mkdir(exist_ok=True)
raster = sc.Raster(opts.grad_raster_time)

## The spoke

Everything here is the readout's own geometry. **The centre sample is not the middle of the ADC
window** — with an even matrix a full spoke is asymmetric by one, which is the same `matrix // 2`
convention `PhaseEncode` uses and what the official reference does.

In [ ]:
spoke = sc.modules.RadialReadout(opts=opts, fov_mm=FOV_MM, matrix=MATRIX, dwell_s=DWELL_S)

print(f'samples          {spoke.num_samples}')
print(f'centre sample    {spoke.center_sample}   (not {spoke.num_samples // 2 - 1}, and not the ADC midpoint)')
print(f'dk               {spoke.dk_per_m:.4f} 1/m  = 1/FOV')
print(f'k along the spoke{spoke.k_first_per_m:9.2f} .. {spoke.k_last_per_m:.2f} 1/m')
print(f'k_max            {spoke.k_max_per_m:.2f} 1/m')
print(f'prephaser        {spoke.prephaser_duration_s * 1e6:.0f} us')
print(f'time to k = 0    {spoke.time_to_center() * 1e3:.3f} ms from the start of the block')

## One repetition, assembled here rather than by a class

Excitation, the spoke at whatever angle the caller wants, a spoiler along that same spoke, and TR
fill. Four lines, and they are the composition a `RadialGRE` module would otherwise hide.

In [ ]:
exc = sc.modules.Excitation(opts=opts, flip_deg=FLIP_DEG, thickness_mm=THICKNESS_MM,
                            duration_s=1e-3)
spoil = sc.modules.spoiler(opts, cycles_per_voxel=4.0, voxel_mm=THICKNESS_MM, axis='z')

START_S = float(raster.ceil(exc().duration))


def repetition(angle_rad, *, phase_deg=0.0, acquire=True):
    """One TR: excite, play the spoke at `angle_rad`, spoil, fill to TR."""
    out = sc.LogicBlock('spoke_tr').add(0.0, exc(phase_deg=phase_deg))
    out.add(START_S, spoke(angle_rad=angle_rad, acquire=acquire, phase_deg=phase_deg))
    tail = START_S + spoke(angle_rad=0.0).duration
    out.add(tail, spoil)
    fill = float(raster.ceil(TR_S - tail - spoil.duration))
    if fill > 1e-9:
        out.add(tail + spoil.duration, pp.make_delay(fill))
    return out


te_s = START_S + spoke.time_to_center() - exc.time_to_center()
print(f'TE {te_s * 1e3:.3f} ms, TR {repetition(0.0).duration * 1e3:.3f} ms')

## Two schedules, one readout

The angles are **data**. Equal increments over π cover k-space uniformly for a fixed spoke count;
the golden angle keeps every prefix of the list approximately uniform, which is what makes it
useful when the scan may be stopped early or binned after the fact.

Neither is the readout's business, and neither changes the spoke.

In [ ]:
def equal_increment(n):
    return [np.pi * i / n for i in range(n)]


def golden_angle(n):
    return [i * np.pi * (3 - np.sqrt(5)) / 2 for i in range(n)]


def phase_deg(n, increment=117.0):
    return 0.5 * increment * n * (n + 1)


def scan(angles):
    out = sc.LogicBlock('radial_gre')
    for n in range(DUMMIES):
        out.add(n * TR_S, repetition(angles[0], phase_deg=phase_deg(n), acquire=False))
    for index, angle in enumerate(angles):
        n = DUMMIES + index
        out.add(n * TR_S, repetition(angle, phase_deg=phase_deg(n)))
    return out


schedules = {'equal': equal_increment(SPOKES), 'golden': golden_angle(SPOKES)}
trees = {name: scan(angles) for name, angles in schedules.items()}
for name, tree in trees.items():
    print(f'{name:7s} {SPOKES} spokes + {DUMMIES} dummies, {tree.duration:.2f} s')

## What was encoded

Measured off the compiled sequence: every spoke straight, through the centre, with the same extent
and spacing, pointing where it was asked to.

In [ ]:
k = sc.kspace(trees['equal'], opts)['k_adc'].reshape(3, SPOKES, spoke.num_samples)
radius = np.linalg.norm(k[:2], axis=0)
wanted = np.degrees(schedules['equal'])
measured = np.degrees(np.arctan2(k[1, :, -1] - k[1, :, 0], k[0, :, -1] - k[0, :, 0]))

print(f'|k| at the centre sample   {np.abs(radius[:, spoke.center_sample]).max():.2e} 1/m')
print(f'angle error                {np.abs((measured - wanted + 90) % 180 - 90).max():.2e} deg')
print(f'kz                         {np.abs(k[2]).max():.2e} 1/m')
print(f'extent spread across spokes{np.ptp(radius.max(axis=1)):.2e} 1/m')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.6))
for ax, (name, tree) in zip(axes, trees.items()):
    traj = sc.kspace(tree, opts)['k_adc'].reshape(3, SPOKES, spoke.num_samples)
    for s in range(SPOKES):
        ax.plot(traj[0, s], traj[1, s], lw=0.6, color=plt.cm.viridis(s / SPOKES))
    ax.plot(0, 0, 'r+', ms=10)
    ax.set(title=f'{name} increment', xlabel='$k_x$ / m$^{-1}$', aspect='equal', xticks=[], yticks=[])
axes[0].set_ylabel('$k_y$ / m$^{-1}$')
fig.suptitle('the same spoke, two schedules -- colour is acquisition order', y=1.0)
fig.tight_layout()

The red cross is `k = 0`, and **every spoke has a sample exactly on it** — which is why the
reference adds a half-sample term to its prephaser, and why this module reports `center_sample`
rather than letting a caller assume `n/2`.

## Centre-out, from the same class

`partial_fourier` spans the family: `1.0` is a full spoke, `0.5` is centre-out, and between them
the spoke is asymmetric about the centre. `Δk` does not move.

In [ ]:
header = ('partial_fourier'.rjust(15) + '  ' + 'samples'.rjust(8) + '  '
          + 'centre'.rjust(7) + '  ' + 'k range / 1/m'.rjust(22) + '  ' + 'dk'.rjust(7))
print(header)
for pf in (1.0, 0.75, 0.5):
    variant = sc.modules.RadialReadout(opts=opts, fov_mm=FOV_MM, matrix=MATRIX,
                                       dwell_s=DWELL_S, partial_fourier=pf)
    print(f'{pf:15.2f}  {variant.num_samples:8d}  {variant.center_sample:7d}  '
          f'{variant.k_first_per_m:10.2f} .. {variant.k_last_per_m:7.2f}  {variant.dk_per_m:7.4f}')

try:
    sc.modules.RadialReadout(opts=opts, fov_mm=FOV_MM, matrix=MATRIX, dwell_s=DWELL_S,
                             partial_fourier=0.25)
except sc.ConfigurationError as error:
    print()
    print(str(error).splitlines()[0])


## The files

In [ ]:
for name, tree in trees.items():
    seq = sc.compile(tree, opts, name=f'radial_gre_{name}', definitions={
        'FOV': [FOV_MM / 1e3, FOV_MM / 1e3, THICKNESS_MM / 1e3],
        'TE': te_s, 'TR': TR_S,
    })
    seq.write(str(SEQ_DIR / f'radial_gre_{name}.seq'))
    print(f'{name:7s} {len(seq.block_events):5d} blocks  {seq.duration()[0]:5.2f} s  '
          f'-> seq/radial_gre_{name}.seq')

## What this notebook established

| | |
|---|---|
| a radial acquisition is **one spoke plus a schedule** | which is why no `RadialGRE` class ships — it would wrap the schedule and own no physics |
| every spoke has a **sample exactly on `k = 0`** | measured, and the module reports which sample rather than leaving a caller to assume |
| the spokes point where they were asked to | angle error below 1e-13 degrees across 64 spokes |
| `partial_fourier` spans full-spoke to centre-out | one parameter, `Δk` unchanged, and below 0.5 it is refused |
| the schedule is data | equal increments and a golden angle out of the same readout instance |

There is no `02_simulate_and_reconstruct.ipynb` here. A radial image needs a non-Cartesian
reconstruction, and the example suite has none to reuse — building a gridding or NUFFT pipeline to
satisfy a checklist would be a new framework rather than an example. The trajectory plot above is
the visual check that a radial acquisition actually needs, and the numbers beside it are what
`tests/modules/test_radial_readout.py` asserts on every commit.